In [10]:
import pandas as pd
import numpy as np

In [11]:
df = pd.read_csv("../data/QM9/data.csv")

In [12]:
rng = np.random.default_rng(10)
nums = rng.choice(df.index, 20)
nums

array([101593, 125072,  34541,  27170, 103692, 108384,  67340,  19530,
       108963,  67089,  20062,  17782,  53760,  90145,  52810, 110125,
         1062,  55668,  68579, 125193])

In [13]:
smiles_list = df.smiles[nums]

In [14]:
from ml_enhance import get_preprocessed_smiles

def molecule_repr(id: str, smiles: str) -> dict:
    return {"id": id, "smiles": smiles}


def generate_json(inputs: list[dict]) -> dict:
    input_file = {
        "options": {
            "max_conformers": 32,
            "max_microstates": 8,
            "metadynamics": False
        },
        "runtime_options": {
        "num_vcpus": 1,
        "batch_size": 5
    },
    }

    input_file.update({"inputs": inputs})

    return input_file

# preprocess the smiles for the quantumFP program
preprocessed_smiles = [get_preprocessed_smiles(smiles) for smiles in smiles_list]

# Filter out all None instances returned from preprocess_smiles
cleaned_smiles = list(filter(lambda x: x not in (None, 'salt', 'atom'), preprocessed_smiles))

inputs = [molecule_repr(idx, smiles) for idx, smiles in enumerate(cleaned_smiles)]

json_obj = generate_json(inputs)

In [15]:
import json

with open("../data/QM9/qm9_test.json", "w") as f:
    json.dump(json_obj, f, indent=4)